### Ingestão de Dados de Reviews do Mercado Livre

Este notebook realiza a ingestão e consolidação de dados de reviews de produtos do Mercado Livre, obtidos de arquivos JSON hospedados em repositórios públicos no GitHub. O processo inclui:

1. **Importação de Bibliotecas**: Utiliza `pandas` para manipulação de dados, `requests` para requisições HTTP (embora não seja usado diretamente aqui) e `os` para operações de sistema de arquivos.

2. **Definição de URLs**: Uma lista de links para arquivos JSON contendo reviews de produtos, validados e prontos para download.

3. **Download e Processamento**: Para cada URL, o código tenta carregar o JSON diretamente usando `pd.read_json()`, adicionando os dados a uma lista de DataFrames temporários. Em caso de erro, registra o problema.

4. **Consolidação**: Se dados forem carregados com sucesso, concatena todos os DataFrames em um único `df_final`, ignorando índices originais para evitar duplicatas.

5. **Salvamento**: Cria o diretório `../data/raw` se necessário e salva o DataFrame consolidado em um arquivo CSV (`reviews_mercadolivre.csv`) com codificação UTF-8 para suportar caracteres especiais.

6. **Relatórios**: Imprime logs de progresso, incluindo sucesso/erro por arquivo, total de reviews consolidados, caminho absoluto do arquivo salvo e lista de colunas encontradas.

Este workflow prepara os dados brutos para análises subsequentes em notebooks posteriores, garantindo integridade e acessibilidade dos dados de reviews. Variáveis como `lines` (exemplo de dados JSON), `response` (resposta HTTP), e URLs individuais podem ser usadas para inspeções adicionais ou debugging.

In [7]:
import pandas as pd
import requests
import os

# Links validados
urls = [
    "https://raw.githubusercontent.com/octaprice/ecommerce-product-dataset/refs/heads/main/data/mercadolivre_com_br/reviews_mercadolivre_com_br_1.json",
    "https://raw.githubusercontent.com/octaprice/ecommerce-product-dataset/refs/heads/main/data/mercadolivre_com_br/reviews_mercadolivre_com_br_2.json"
]

all_dfs = []

print("🚀 Iniciando ingestão (Formato Lista JSON)...")

for url in urls:
    try:
        # Para listas JSON padrão, o read_json funciona direto se apontarmos o link raw
        df_temp = pd.read_json(url)
        all_dfs.append(df_temp)
        print(f"✅ Sucesso: {url.split('/')[-1]} | {len(df_temp)} reviews.")
    except Exception as e:
        print(f"❌ Erro ao processar {url.split('/')[-1]}: {e}")

if all_dfs:
    df_final = pd.concat(all_dfs, ignore_index=True)
    
    # Criar pastas
    os.makedirs('../data/raw', exist_ok=True)
    
    # Salvar em CSV para o Notebook 02
    caminho_csv = '../data/raw/reviews_mercadolivre.csv'
    df_final.to_csv(caminho_csv, index=False, encoding='utf-8-sig')
    
    print("-" * 40)
    print(f"📊 TOTAL CONSOLIDADO: {len(df_final)} reviews.")
    print(f"📁 SALVO EM: {os.path.abspath(caminho_csv)}")
    print(f"📝 COLUNAS ENCONTRADAS: {df_final.columns.tolist()}")
else:
    print("‼️ Nenhum dado foi carregado.")

🚀 Iniciando ingestão (Formato Lista JSON)...
✅ Sucesso: reviews_mercadolivre_com_br_1.json | 103476 reviews.
✅ Sucesso: reviews_mercadolivre_com_br_2.json | 103476 reviews.
----------------------------------------
📊 TOTAL CONSOLIDADO: 206952 reviews.
📁 SALVO EM: h:\Meu Drive\DEV\poc-gliner2\data\raw\reviews_mercadolivre.csv
📝 COLUNAS ENCONTRADAS: ['date', 'rating', 'content', 'product_url']
